In [1]:
import pandas as pd

train_data = pd.read_parquet("data/block1_train.parquet", engine='fastparquet')
test_data  = pd.read_parquet("data/block2_test.parquet", engine='fastparquet')

In [2]:
target_cols = [f"Insurer_{chr(i)}_price" for i in range(ord('A'), ord('K') + 1)]
deductible_cols = [f"Insurer_{chr(i)}_deductible" for i in range(ord('A'), ord('K') + 1)]

In [3]:
import pandas as pd
import numpy as np

def detect_categorical_columns(df):
    categorical_cols = []
    
    for col in df.columns:
        # Uzimamo samo vrednosti koje nisu NaN za testiranje
        non_null_values = df[col].dropna()
        
        if non_null_values.empty:
            # Ako je cela kolona prazna, tretiramo je kao kategorijsku ili je brišemo
            categorical_cols.append(col)
            continue
            
        try:
            # Pokušavamo da konvertujemo celu kolonu (bez NaN) u float
            pd.to_numeric(non_null_values, errors='raise')
        except (ValueError, TypeError):
            # Ako baci grešku, znači da ima teksta koji nije broj
            categorical_cols.append(col)
            
    return categorical_cols

# Korišćenje:
cat_cols = detect_categorical_columns(train_data)
print(cat_cols)

['vehicle_number_plate', 'coverage', 'payment_frequency', 'contractor_birthdate', 'is_driver_owner', 'usage', 'second_driver_birthdate', 'second_driver_claim_free_years', 'vehicle_maker', 'vehicle_model', 'vehicle_fuel_type', 'vehicle_primary_color', 'vehicle_first_registration_date', 'vehicle_country_first_registration_date', 'vehicle_last_registration_date', 'vehicle_inspection_report_date', 'vehicle_inspection_expiry_date', 'vehicle_odometer_verdict_code', 'vehicle_is_imported', 'vehicle_is_imported_within_last_12_months', 'vehicle_can_be_registered', 'vehicle_has_open_recall', 'vehicle_is_marked_for_export', 'vehicle_is_taxi', 'province', 'municipality', 'postal_code_houses_owned_by_rental_association_ratio']


In [4]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

def encode_categorical(train, test, cat_cols, high_sparsity_threshold=10):

    train_enc = train.copy()
    test_enc = test.copy()
    
    high_sparse_cols = [c for c in cat_cols if train[c].nunique() > high_sparsity_threshold]
    low_sparse_cols = [c for c in cat_cols if train[c].nunique() <= high_sparsity_threshold]
    
    # Ordinal encoding za high sparsity
    if high_sparse_cols:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        train_enc[high_sparse_cols] = oe.fit_transform(train[high_sparse_cols])
        test_enc[high_sparse_cols] = oe.transform(test[high_sparse_cols])
    
    # One-hot encoding za low sparsity
    if low_sparse_cols:
        ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        train_ohe = ohe.fit_transform(train[low_sparse_cols])
        test_ohe = ohe.transform(test[low_sparse_cols])
        
        # Napravimo kolone sa imenima
        ohe_cols = ohe.get_feature_names_out(low_sparse_cols)
        train_ohe_df = pd.DataFrame(train_ohe, columns=ohe_cols, index=train.index)
        test_ohe_df = pd.DataFrame(test_ohe, columns=ohe_cols, index=test.index)
        
        # Drop original low sparsity cols i dodaj one-hot
        train_enc = train_enc.drop(columns=low_sparse_cols).join(train_ohe_df)
        test_enc = test_enc.drop(columns=low_sparse_cols).join(test_ohe_df)
    
    return train_enc, test_enc

In [5]:
train_data_enc, test_data_enc = encode_categorical(train_data, test_data, cat_cols)

In [6]:
object_cols = train_data_enc.select_dtypes(include=['object']).columns

In [7]:
# 1. Uzimamo sve kolone koje su trenutno tipa 'object'
object_cols = train_data_enc.select_dtypes(include=['object']).columns

print(f"Započinjem konverziju {len(object_cols)} kolona...")

for col in object_cols:
    # Konvertujemo u numerik (float64 po defaultu)
    # errors='coerce' sprečava pucanje koda ako naiđe na tekstualni bag
    train_data_enc[col] = pd.to_numeric(train_data_enc[col], errors='coerce')

    # Isto radimo i za test set da bi struktura ostala identična
    if col in test_data_enc.columns:
        test_data_enc[col] = pd.to_numeric(test_data_enc[col], errors='coerce')

# 2. Finalna provera tipova
print("-" * 30)
print("Konverzija gotova!")
print(f"Preostalo 'object' kolona u train: {len(train_data_enc.select_dtypes(include=['object']).columns)}")

Započinjem konverziju 105 kolona...
------------------------------
Konverzija gotova!
Preostalo 'object' kolona u train: 0


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# 1. Podela podataka pre petlje
X = train_data_enc.copy()
Y = train_data[target_cols]

# Delimo na Train (za trening) i Val (za proveru)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

trained_models = {}

print("Počinjem trening po targetima...")

for col in target_cols:
    print(f"--- Treniram model za: {col} ---")
    
    # Koristimo samo redove gde trenutni target nije NaN (unutar trening seta)
    valid_mask = Y_train[col].notna()
    
    X_train_sub = X_train[valid_mask]
    y_train_sub = Y_train.loc[valid_mask, col]
    
    # 2. Model
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        n_jobs=-1  # Koristi sve CPU jezgre za brzinu
    )
    
    # Trening na trenutno dostupnim feature-ima
    model.fit(X_train_sub, y_train_sub)
    trained_models[col] = model
    
    # 3. KLJUČNI KORAK: Predviđanje za oba seta
    # Svako predviđanje postaje novi feature za naredne targete u petlji
    X_train[f"pred_{col}"] = model.predict(X_train)
    X_val[f"pred_{col}"] = model.predict(X_val)
    
    print(f"Model za {col} je završen i dodat kao feature.")

print("\nSvi modeli su istrenirani!")

Počinjem trening po targetima...
--- Treniram model za: Insurer_A_price ---
Model za Insurer_A_price je završen i dodat kao feature.
--- Treniram model za: Insurer_B_price ---
Model za Insurer_B_price je završen i dodat kao feature.
--- Treniram model za: Insurer_C_price ---
Model za Insurer_C_price je završen i dodat kao feature.
--- Treniram model za: Insurer_D_price ---
Model za Insurer_D_price je završen i dodat kao feature.
--- Treniram model za: Insurer_E_price ---
Model za Insurer_E_price je završen i dodat kao feature.
--- Treniram model za: Insurer_F_price ---
Model za Insurer_F_price je završen i dodat kao feature.
--- Treniram model za: Insurer_G_price ---
Model za Insurer_G_price je završen i dodat kao feature.
--- Treniram model za: Insurer_H_price ---
Model za Insurer_H_price je završen i dodat kao feature.
--- Treniram model za: Insurer_I_price ---
Model za Insurer_I_price je završen i dodat kao feature.
--- Treniram model za: Insurer_J_price ---
Model za Insurer_J_price

In [60]:
results = []

for col in target_cols:
    # Predviđanja smo već sačuvali u X_val tokom petlje pod imenom f"pred_{col}"
    y_pred = X_val[f"pred_{col}"]
    y_true = Y_val[col]
    
    # Filtriraj NaN vrednosti iz Y_val (ako ih ima)
    mask = y_true.notna()
    
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    r2 = r2_score(y_true[mask], y_pred[mask])
    
    results.append({
        'Target': col,
        'MAE': mae,
        'R2': r2
    })

# Prikaz rezultata u tabeli
df_results = pd.DataFrame(results)
print("\n--- FINALNI REZULTATI NA VALIDACIONOM SETU ---")
print(df_results)

print(f"\nProsečan MAE svih targeta: {df_results['MAE'].mean():.4f}")


--- FINALNI REZULTATI NA VALIDACIONOM SETU ---
             Target       MAE        R2
0   Insurer_A_price  1.406567  0.962417
1   Insurer_B_price  0.522485  0.997913
2   Insurer_C_price  0.453745  0.998024
3   Insurer_D_price  0.492064  0.998424
4   Insurer_E_price  0.828891  0.993043
5   Insurer_F_price  0.503142  0.997808
6   Insurer_G_price  0.320575  0.998668
7   Insurer_H_price  0.455044  0.998528
8   Insurer_I_price  0.474576  0.997834
9   Insurer_J_price  0.603055  0.995542
10  Insurer_K_price  0.573585  0.998147

Prosečan MAE svih targeta: 0.6031


In [ ]:
import pandas as pd
from xgboost import XGBRegressor

# 1. PRIPREMA: Izbacujemo sve targete iz trening seta pre početka
# Moramo izbaciti 'Insurer_A_price', 'Insurer_B_price'... iz X setova
X_train_current = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
X_test_current = test_data_enc.drop(columns=target_cols, errors='ignore').copy()

trained_models = {}
test_predictions = {}

print("Započeto treniranje...")

for col in target_cols:
    print(f"Obrađujem: {col}")
    
    # Maska za validne redove ostaje ista jer gledamo Y_train (originalne targete)
    valid_mask = Y_train[col].notna()
    
    # Uzimamo trenutne feature (oni sada NE sadrže originalne cene, samo predikcije)
    X_sub = X_train_current.loc[valid_mask]
    y_sub = Y_train.loc[valid_mask, col]
    
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    
    # Fit modela na čistim podacima
    model.fit(X_sub, y_sub)
    trained_models[col] = model
    
    # DODAVANJE PREDVIĐANJA KAO FEATURE:
    # Ovo je jedini način na koji sledeći model "vidi" cene prethodnih
    train_preds = model.predict(X_train_current)
    X_train_current[f"pred_{col}"] = train_preds
    
    test_preds = model.predict(X_test_current)
    X_test_current[f"pred_{col}"] = test_preds
    
    test_predictions[col] = np.round(test_preds, 2)

submission = pd.DataFrame(test_predictions)

submission.insert(0, 'quote_id', test_data['quote_id'].astype(int).values)

submission.to_csv('finalni_rezultati.csv', index=False, sep=';')

print("\n--- PROCES ZAVRŠEN ---")
print("Prvih par redova tvog submission-a:")
print(submission.head())

Započeto treniranje...
Obrađujem: Insurer_A_price
Obrađujem: Insurer_B_price
Obrađujem: Insurer_C_price
Obrađujem: Insurer_D_price
Obrađujem: Insurer_E_price
Obrađujem: Insurer_F_price
Obrađujem: Insurer_G_price
Obrađujem: Insurer_H_price
Obrađujem: Insurer_I_price
Obrađujem: Insurer_J_price
Obrađujem: Insurer_K_price

--- PROCES ZAVRŠEN ---
Prvih par redova tvog submission-a:
   quote_id  Insurer_A_price  Insurer_B_price  Insurer_C_price  \
0    541293        56.320000        76.529999        61.439999   
1    541294        28.969999        29.510000        36.090000   
2    541295        27.980000        37.169998        33.099998   
3    541296       269.459991       334.519989       292.200012   
4    541297       167.550003       236.279999       225.119995   

   Insurer_D_price  Insurer_E_price  Insurer_F_price  Insurer_G_price  \
0        96.320000        95.480003        61.410000        60.110001   
1        37.389999        44.639999        34.639999        28.160000   
2   

In [16]:
test_data.shape

(164092, 144)

In [18]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# 1. Priprema podataka
# Izbacujemo targete (cene) iz X seta pre podele, jer njih model ne sme da vidi kao ulaz
X = train_data_enc.drop(columns=target_cols, errors='ignore').copy()
Y = train_data[target_cols]

# Podela na Train i Val
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

trained_models = {}

print("Počinjem trening po targetima (sa chaining metodom)...")

for col in target_cols:
    print(f"--- Treniram model za: {col} ---")
    
    # Koristimo samo redove gde trenutni target u Y_train nije NaN
    valid_mask = Y_train[col].notna()
    
    # Uzimamo trenutne feature (originalni + predviđanja prethodnih modela)
    X_train_sub = X_train.loc[valid_mask]
    y_train_sub = Y_train.loc[valid_mask, col]
    
    # 2. Model definicija
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )
    
    # Trening
    model.fit(X_train_sub, y_train_sub)
    trained_models[col] = model
    
    # 3. DODAVANJE PREDVIĐANJA KAO NOVI FEATURE
    # Predviđamo za CEO X_train i X_val kako bi sledeći model imao ove podatke
    X_train[f"pred_{col}"] = model.predict(X_train)
    X_val[f"pred_{col}"] = model.predict(X_val)
    
    print(f"Model za {col} završen. Dodat feature 'pred_{col}' u Train i Val.")

print("\nSvi modeli su istrenirani! Sada možeš proveriti metrike.")

# --- OPCIONA PROVERA REZULTATA ---
results = []
for col in target_cols:
    y_true = Y_val[col]
    y_pred = X_val[f"pred_{col}"]
    mask = y_true.notna()
    
    mae = mean_absolute_error(y_true[mask], y_pred[mask])
    results.append({'Target': col, 'MAE': mae})

print("\nMAE po osiguravačima na validacionom setu:")
print(pd.DataFrame(results))

Počinjem trening po targetima (sa chaining metodom)...
--- Treniram model za: Insurer_A_price ---
Model za Insurer_A_price završen. Dodat feature 'pred_Insurer_A_price' u Train i Val.
--- Treniram model za: Insurer_B_price ---
Model za Insurer_B_price završen. Dodat feature 'pred_Insurer_B_price' u Train i Val.
--- Treniram model za: Insurer_C_price ---
Model za Insurer_C_price završen. Dodat feature 'pred_Insurer_C_price' u Train i Val.
--- Treniram model za: Insurer_D_price ---
Model za Insurer_D_price završen. Dodat feature 'pred_Insurer_D_price' u Train i Val.
--- Treniram model za: Insurer_E_price ---
Model za Insurer_E_price završen. Dodat feature 'pred_Insurer_E_price' u Train i Val.
--- Treniram model za: Insurer_F_price ---
Model za Insurer_F_price završen. Dodat feature 'pred_Insurer_F_price' u Train i Val.
--- Treniram model za: Insurer_G_price ---
Model za Insurer_G_price završen. Dodat feature 'pred_Insurer_G_price' u Train i Val.
--- Treniram model za: Insurer_H_price ---

In [19]:
print(pd.DataFrame(results))

             Target        MAE
0   Insurer_A_price  12.445129
1   Insurer_B_price  11.612943
2   Insurer_C_price  10.796713
3   Insurer_D_price  12.269809
4   Insurer_E_price  18.605483
5   Insurer_F_price  11.956905
6   Insurer_G_price   7.447456
7   Insurer_H_price  15.039967
8   Insurer_I_price  16.337037
9   Insurer_J_price  13.882767
10  Insurer_K_price  13.787913
